### Chaimaa Faouzi, Master Génie Logiciel pour le Cloud Computing

### * Algorithme KNN : 

In [3]:
import numpy as np  # bibliothèque pour calculs numériques

# 1. Données d'entraînement
# X = caractéristiques (ex : notes en math et physique)
X = np.array([
    [10, 12],
    [12, 14],
    [14, 15],
    [16, 18],
    [18, 20],
    [20, 22]
])
# y = classes (0 = non admis, 1 = admis)
y = np.array([0, 0, 0, 1, 1, 1])

# 2. Fonction distance euclidienne
# Cette fonction calcule la distance entre deux points, Elle est générale (peut être utilisée partout)
def distance(a, b):
    # calcule la distance entre deux points
    return np.sqrt(np.sum((a - b) ** 2))

# 3. Algorithme KNN (outil de prédiction)
# Cette fonction est seulement définie ici, Elle ne sera utilisée que plus tard
def knn(X_train, y_train, point, k):
    distances = []  # liste pour stocker (distance, classe)

    # 3.1 calculer la distance entre le point à prédire et chaque point d'entraînement
    for i in range(len(X_train)):
        d = distance(point, X_train[i])
        distances.append((d, y_train[i]))

    # 3.2 trier les distances du plus proche au plus loin
    distances.sort(key=lambda x: x[0])

    # 3.3 prendre les k plus proches voisins
    voisins = distances[:k]

    # 3.4 récupérer les classes des voisins
    classes = [v[1] for v in voisins]

    # 3.5 vote majoritaire (classe la plus fréquente)
    prediction = max(set(classes), key=classes.count)
    return prediction

# 4. Validation croisée (k-fold)
# Cette fonction va TESTER plusieurs fois KNN, pour évaluer la qualité d'un k donné
def cross_validation(X, y, k_voisins, k_folds):
    n = len(X)  # nombre total d'échantillons
    fold_size = n // k_folds  # taille de chaque bloc
    scores = []  # stockage des résultats

    # 4.1 boucle pour chaque fold
    for i in range(k_folds):
        # définir la partie test
        start = i * fold_size
        end = start + fold_size

        # 4.2 créer données de test
        X_test = X[start:end]
        y_test = y[start:end]

        # 4.3 créer données d'entraînement (tout sauf test)
        X_train = np.concatenate((X[:start], X[end:]))
        y_train = np.concatenate((y[:start], y[end:]))
        correct = 0  # compteur bonnes prédictions

        # 4.4 tester chaque point du test
        for j in range(len(X_test)):
            pred = knn(X_train, y_train, X_test[j], k_voisins)  # ICI on utilise KNN

            # comparer prédiction avec vraie valeur
            if pred == y_test[j]:
                correct += 1

        # 4.5 calcul de la précision pour ce fold
        accuracy = correct / len(X_test)
        scores.append(accuracy)

    # 4.6 moyenne des scores
    return sum(scores) / len(scores)

# 5. Recherche du meilleur k 
# C'est ici que l'exécution commence vraiment, On teste plusieurs valeurs de k
k_values = [1, 3, 5]
best_k = None
best_score = 0

# tester chaque k
for k in k_values:

    # 5.1 tester k avec validation croisée
    score = cross_validation(X, y, k_voisins=k, k_folds=3)
    print("k =", k, "-> score =", score)

    # 5.2 garder le meilleur k
    if score > best_score:
        best_score = score
        best_k = k

print("\nMeilleur k trouvé :", best_k)

# 6. Utilisation finale du KNN
# Maintenant on utilise le meilleur k trouvé
nouveau_point = np.array([15, 16]) 

# prédiction avec le meilleur k
resultat = knn(X, y, nouveau_point, best_k)

# affichage du résultat
if resultat == 1:
    print("Résultat final du point [15,16] : Admis")
else:
    print("Résultat final du point [15,16] : Non admis")

k = 1 -> score = 1.0
k = 3 -> score = 0.3333333333333333
k = 5 -> score = 0.16666666666666666

Meilleur k trouvé : 1
Résultat final du point [15,16] : Non admis


### * Algorithme SVM : 

In [8]:
# 1. Données
# X : caractéristiques (ex : notes en math et physique)
X = np.array([[10, 12], [12, 14], [14, 15], [16, 18], [18, 20], [20, 22]])

# Y : -1 = non admis, +1 = admis
y = np.array([-1, -1, -1, 1, 1, 1])

# 2. Initialisation des paramètres
# w : vecteur de poids (définit la direction de la frontière)
w = np.zeros(X.shape[1])  # initialisé à 0 → [0, 0]

# b : biais (déplace la frontière)
b = 0

# learning_rate : vitesse d’apprentissage (petits pas)
learning_rate = 0.001

# epochs : nombre de répétitions de l’apprentissage
epochs = 1000

# 3. Phase d'apprentissage 
# Le modèle va ajuster w et b pour bien séparer les classes
for epoch in range(epochs):  # répéter plusieurs fois

    for i in range(len(X)):  # parcourir chaque point

        # calcul de la fonction (w.x + b) de décision qui permet de séparer les classes (-1 et +1) et maximiser la marge
        # np.dot = produit scalaire
        condition = y[i] * (np.dot(X[i], w) + b)

        # CAS 1 : point bien classé (dans la bonne zone)
        if condition >= 1:
            # Mise à jour de w pour réduire sa norme, ce qui permet indirectement d’augmenter la marge du SVM.
            w = w - learning_rate * (2 * w)

        # CAS 2 : point mal classé ou trop proche
        else:
            # Correction des erreurs : ajuste w et b pour mieux classifier les points mal classés et rapprocher la frontière des bonnes décisions.
            w = w - learning_rate * (2 * w - y[i] * X[i])
            b = b - learning_rate * (-y[i])

# 4. Fonction de prédiction
# pour prédire la position d’un point par rapport à la frontière de décision
def predict(X):
    # calcul de (w.x + b)
    result = np.dot(X, w) + b
    return np.sign(result)
    # np.sign :
    # > 0 → +1 (admis)
    # < 0 → -1 (non admis)

# 5. On teste un nouvel étudiant
nouveau_point = np.array([15, 16])

# prédiction
prediction = predict(nouveau_point)

# 6. Affichage du résultat
if prediction == 1:
    print("Admis")
else:
    print("Non admis")   

Admis
